# Практическая работа № 8 (пример выполнения)
## Тестирование, рефакторинг и подготовка к защите «Тайны исчезнувшего дива»

**Курс:** ООП для разработки компьютерных игр
**Модуль:** 8 — Тестирование, рефакторинг, подготовка к защите
**Проект:** «Тайна исчезнувшего дива» — мистический детектив по мотивам цикла Виктора Дашкевича о графе Аверине
**Формат:** демонстрационный блокнот с полным разбором задания
**Цель:** показать, как довести проект до защиты: тесты, моки, рефакторинг, документация

---

## 📖 Контекст

После модулей 1–7 проект готов функционально. Осталось отполировать: тесты, чистота кода, документация, презентация.

## 🎯 Что демонстрирует блокнот

| Тема | Что показываем |
|---|---|
| **pytest** | fixtures, parametrize |
| **Mocking** | Mock, monkeypatch |
| **TDD** | Red-Green-Refactor |
| **Рефакторинг** | удаление дублирования, константы |
| **Защита** | структура презентации |

## Ключевая идея

Тесты дают смелость менять код, рефакторинг делает код чище, документация и презентация — продают результат.

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

---
# Шаг 1. Тесты: fixtures и parametrize

In [ ]:
import pytest


class Detective:
    def __init__(self, name, health=100):
        self.name = name
        self.health = health

    def take_damage(self, amount):
        self.health = max(0, self.health - amount)

    def is_alive(self):
        return self.health > 0


@pytest.fixture
def detective():
    return Detective("Аверин", health=100)


def test_take_damage(detective):
    detective.take_damage(30)
    assert detective.health == 70


def test_damage_clamps_at_zero(detective):
    detective.take_damage(999)
    assert detective.health == 0


@pytest.mark.parametrize("hp,expected", [(100, True), (0, False), (-5, False)])
def test_is_alive(hp, expected):
    assert Detective("x", hp).is_alive() is expected

**Что мы видим:** fixture готовит детектива, parametrize прогоняет три набора данных одним тестом. `take_damage` корректно ограничивает здоровье снизу.

---
# Шаг 2. Mocking

In [ ]:
from unittest.mock import Mock


class AudioManager:
    def play(self, name):
        # в реальном проекте — обращение к звуковой карте
        raise RuntimeError("нет звуковой карты")


def notify_player_audio(audio):
    audio.play("victory")


# В тесте реальный AudioManager заменяем моком
audio = Mock()
notify_player_audio(audio)

audio.play.assert_called_once_with("victory")
print("Мок подтвердил вызов play('victory') без звуковой карты")

**Что мы видим:** мок заменяет `AudioManager`, и тест проверяет, что метод вызван с нужным аргументом — без обращения к реальному оборудованию.

---
# Шаг 3. TDD: Red-Green-Refactor

In [ ]:
# Red: пишем тест на ещё не существующий Inventory


class Inventory:
    def __init__(self):
        self.items = []

    def add(self, item):
        self.items.append(item)

    def has(self, item):
        return item in self.items


# Green: тест проходит с минимальной реализацией
def test_inventory_add_and_has():
    inv = Inventory()
    inv.add("лупа")
    assert inv.has("лупа")
    assert not inv.has("фонарь")


test_inventory_add_and_has()
print("TDD-цикл: тест прошёл ✅")

**Что мы видим:** сначала контракт (тест), потом минимальная реализация. Тест фиксирует ожидаемое поведение `Inventory`.

---
# Шаг 4. Рефакторинг

In [ ]:
# ДО: дублирование и магическое число


def damage_enemy(enemy):
    enemy.hp -= 10
    if enemy.hp < 0:
        enemy.hp = 0


def damage_player(player):
    player.hp -= 10
    if player.hp < 0:
        player.hp = 0


# ПОСЛЕ: Extract Method + константа
DAMAGE = 10


def apply_damage(target, amount=DAMAGE):
    target.hp = max(0, target.hp - amount)


class Unit:
    def __init__(self, hp):
        self.hp = hp


unit = Unit(5)
apply_damage(unit)
print(f"После урона: hp={unit.hp} (ограничено снизу нулём)")

**Что мы видим:** дублирование устранено, магическое число стало константой, логика ограничения вынесена в одну функцию. Поведение не изменилось.

---
# Шаг 5. Документация

## 5.1. README (минимум)

```markdown
# Тайна исчезнувшего дива

Мистический детектив на PyGame. Расследуйте исчезновение дива,
управляйте детективом и Кузей.

## Запуск
pip install -r requirements.txt
python main.py

## Тесты
pytest -v

## Структура
domain/       — правила игры, ECS
application/  — Game, GameLoop, системы
infrastructure/ — звук, конфиг, загрузка
presentation/ — отрисовка, UI, ввод

## Паттерны
State, Strategy, Command, Observer, Factory, Singleton + ECS.
```

## 5.2. ADR-008

```markdown
# ADR-008: тестирование и финализация

## Контекст
- Ключевая логика покрыта частично;
- Есть дублирование и магические числа.

## Решение
- pytest с fixtures и parametrize;
- моки для внешних зависимостей;
- рефакторинг дублирования.

## Последствия
- ✅ Регрессии ловятся сразу.
- ✅ Код чище, тесты быстрее.
```

## 5.3. UML
Добавьте диаграммы классов, состояний, слоёв и ECS в `docs/`.

---
# Шаг 6. Подготовка презентации

## 6.1. Слайды

1. **Титульный** — название, автор, тема.
2. **Идея** — мистический детектив, механики.
3. **Демонстрация** — живой запуск.
4. **Архитектура** — слои, направление зависимостей.
5. **Паттерны** — какие и где (State, Command, Observer, Factory, Singleton).
6. **ECS** — компоненты и системы.
7. **Тестирование** — что покрыто, моки.
8. **Итоги** — сложные решения, выводы.

## 6.2. Сценарий демо (3 минуты)

1. Запуск игры (15 с).
2. Меню → расследование (30 с).
3. Кузя находит улику → событие в EventBus (45 с).
4. Отмена хода через Command (30 с).
5. Победа → VictoryState (30 с).
6. Показ структуры кода (30 с).

## 6.3. Вопросы для самопроверки

- Почему Observer, а не прямые вызовы?
- Где границы слоёв и почему?
- Что покрыто тестами, что нет и почему?

---
# Итоги

## Что сделали

| Тема | Результат |
|---|---|
| **pytest** | fixtures, parametrize |
| **Mocking** | Mock, без реального I/O |
| **TDD** | Red-Green-Refactor |
| **Рефакторинг** | дублирование и магические числа убраны |
| **Защита** | README, ADR, презентация, сценарий демо |

## Что это даёт итоговому проекту

- **Тестирование (8 баллов):** тесты покрывают ключевую логику, моки ускоряют.
- **Документация (5 баллов):** README, ADR, UML.
- **Демонстрация и презентация (5 баллов):** структура и сценарий.
- **Код-ревью (5 баллов):** готовность объяснять решения.

## Литература

**Базовая:**
- Документация pytest.
- Beck K. «Test-Driven Development by Example».

**Углублённая:**
- Fowler M. «Refactoring».
- [Refactoring Guru: smells](https://refactoring.guru/refactoring/smells).
- [Google Engineering Practices: Code Review](https://google.github.io/eng-practices/review/).